# Lab 3 — Quantize & Optimize LLMs (bitsandbytes: INT8 and NF4)

The memory arithmetic of LLM inference is unforgiving:

| Precision | Bytes/param | Llama 3 8B weights | Llama 3 70B weights |
|-----------|-------------|---------------------|----------------------|
| fp32 | 4 | 32 GB | 280 GB |
| fp16 / bf16 | 2 | 16 GB | 140 GB |
| **INT8** | 1 | **8 GB** | 70 GB |
| **INT4 / NF4** | ~0.5 | **4 GB** | 35 GB |

Your RTX 3060 Ti has 8 GB of VRAM. In fp16 it cannot hold Llama 3 8B (the weights alone need 16 GB). In **NF4**, it can hold the weights *and* leave room for KV cache and batching. Quantization is what turns "you need H100s to run this" into "your laptop will do".

### The three key quantization papers

- **[LLM.int8() (Dettmers et al., 2022)](https://arxiv.org/abs/2208.07339)** — the paper that made 8-bit inference practical for LLMs. Introduces mixed-precision decomposition that handles outlier activations, giving ~zero-degradation INT8.
- **[QLoRA (Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314)** — introduces **NF4** (NormalFloat-4), a 4-bit data type tailored for normally-distributed weights. Got a 65B model fine-tuned on a single 48GB GPU. Core lab infrastructure at every open fine-tuning shop.
- **[GPTQ (Frantar et al., 2022)](https://arxiv.org/abs/2210.17323)** — post-training quantization with second-order information. Better accuracy/compression than naive rounding. See also **[AWQ (Lin et al., 2023)](https://arxiv.org/abs/2306.00978)** for the activation-aware variant.

### What you'll measure

Using **`TinyLlama-1.1B-Chat`** (1.1B params, open, no auth, ~2.2 GB in fp16), we'll load it in three formats and compare:

1. **VRAM footprint** — fp16 vs INT8 vs NF4
2. **Generation latency** — is int8 actually slower? (yes, sometimes, on consumer GPUs)
3. **Quality** — measured by perplexity on held-out English. NF4 is typically within 1% of fp16

### When to use which in production

| Goal | Choose |
|------|--------|
| Fastest possible inference | fp16 (or fp8 on H100) |
| Fit big model on small GPU | **NF4** for weights, fp16 activations |
| Calibration-free quantization | bitsandbytes `load_in_4bit` with NF4 |
| Best accuracy at 4-bit | **GPTQ / AWQ** (requires calibration data) |
| Production serving at scale | TensorRT-LLM or vLLM with INT8/FP8 (best kernels) |

We're doing the bitsandbytes flavour here because it's the one you call with **one extra argument** — no calibration, no compilation. It's what every open fine-tuning / QLoRA / inference demo uses.

---

## Step 1 — Load TinyLlama in fp16 (the baseline)

In [1]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

device = torch.device('cuda')
MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

def vram_of(model):
    """Sum parameter + buffer bytes, report in GB."""
    bytes_ = sum(p.numel() * p.element_size() for p in model.parameters())
    bytes_ += sum(b.numel() * b.element_size() for b in model.buffers())
    return bytes_ / 1e9

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Loading fp16 baseline...')
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
vram_fp16 = vram_of(model_fp16)
print(f'fp16 model weights: {vram_fp16:.2f} GB on GPU')
print(f'  (TinyLlama has {model_fp16.num_parameters()/1e9:.2f}B params × 2 bytes ≈ {model_fp16.num_parameters()*2/1e9:.2f} GB theoretical)')

/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

fp16 model weights: 2.20 GB on GPU
  (TinyLlama has 1.10B params × 2 bytes ≈ 2.20 GB theoretical)


In [2]:
from preporato_labs import Lab
lab = Lab('quantization')
lab.check(1)

OK — fp16 TinyLlama-1.1B uses ~2.20 GB of VRAM
STEP_PASSED


Step 1 Complete! Scroll down to continue...

True

## Step 2 — Load in INT8 and NF4 with bitsandbytes

`BitsAndBytesConfig` is HuggingFace's one-line interface to bitsandbytes. Pass it to `from_pretrained`:

### INT8 (`load_in_8bit=True`)

Implements the LLM.int8() paper: most linear layers run in INT8, with a fp16 fallback for outlier features that would otherwise catastrophically lose precision. Zero-degradation on most benchmarks. Half the VRAM of fp16.

### NF4 (`load_in_4bit=True` + `bnb_4bit_quant_type='nf4'`)

The NormalFloat-4 format from the QLoRA paper. Encodes each weight in 4 bits using 16 levels chosen to match the quantiles of a standard normal distribution (which is approximately how Transformer weights are distributed). ~1/4 the VRAM of fp16 with surprisingly small quality loss.

There's also an `fp4` option, slightly simpler encoding; NF4 is the default and usually wins.

### Double quantization (`bnb_4bit_use_double_quant=True`)

Optional: even the quantization *constants* get quantized, saving another ~0.4 bits/param. Free win, always on.

In [3]:
# Free up VRAM from fp16 model before loading quantized variants
# (we'll reload fp16 later for the generation benchmark)
print('Loading INT8 model...')
bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_int8, device_map='cuda')
vram_int8 = vram_of(model_int8)
print(f'INT8 weights: {vram_int8:.2f} GB')

# Free INT8, load NF4
del model_int8
torch.cuda.empty_cache()

print('Loading NF4 model...')
bnb_nf4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model_nf4 = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_nf4, device_map='cuda')
vram_nf4 = vram_of(model_nf4)
print(f'NF4 weights: {vram_nf4:.2f} GB')

Loading INT8 model...
INT8 weights: 1.23 GB
Loading NF4 model...
NF4 weights: 0.75 GB


In [4]:
print(f'\n=== VRAM comparison ===')
print(f'fp16 : {vram_fp16:.2f} GB   (baseline)')
print(f'int8 : {vram_int8:.2f} GB   ({100*vram_int8/vram_fp16:.0f}% of fp16)')
print(f'nf4  : {vram_nf4:.2f} GB   ({100*vram_nf4/vram_fp16:.0f}% of fp16)')
print()
print('Takeaway for an 8GB GPU: NF4 lets you hold a ~30B-param model, INT8 a ~15B model,')
print('and fp16 a ~3.5B model. Which is why every local-inference stack (Ollama, llama.cpp,')
print('LM Studio) ships 4-bit quantizations of everything.')


=== VRAM comparison ===
fp16 : 2.20 GB   (baseline)
int8 : 1.23 GB   (56% of fp16)
nf4  : 0.75 GB   (34% of fp16)

Takeaway for an 8GB GPU: NF4 lets you hold a ~30B-param model, INT8 a ~15B model,
and fp16 a ~3.5B model. Which is why every local-inference stack (Ollama, llama.cpp,
LM Studio) ships 4-bit quantizations of everything.


In [7]:
lab.check(2)

OK — VRAM: fp16=2.20 GB -> int8=1.23 GB (56%) -> nf4=0.75 GB (34%)
STEP_PASSED


Step 2 Complete! Scroll down to continue...

True

## Step 3 — Latency benchmark across precisions

Quantized models use less memory. But are they *faster*?

On server GPUs (A100, H100) with native INT8 tensor cores, the answer is: **yes**, INT8 can be 2× faster than fp16 at serving. On consumer GPUs like the 3060 Ti, the story is murkier — bitsandbytes has to dequantize weights back to fp16 on the fly for matmuls, and the dequant overhead can eat the gain.

**The dominant reason to quantize on consumer GPUs is still memory, not throughput.** If the model doesn't fit in VRAM in fp16, nothing else matters. With plenty of VRAM, fp16 is usually the latency winner.

We'll benchmark all three on the same 50-token generation.

In [5]:
# Free the NF4 model first to have room for a fresh fp16 load for bench
del model_fp16
del model_nf4
torch.cuda.empty_cache()

prompt = 'The future of artificial intelligence is'
input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to('cuda')
GEN_TOKENS = 50

def bench(model, label, n_trials=3):
    times = []
    # Warmup
    _ = model.generate(input_ids, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    for _ in range(n_trials):
        t0 = time.perf_counter()
        _ = model.generate(input_ids, max_new_tokens=GEN_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    ms_per_token = (sum(times) / len(times)) / GEN_TOKENS
    print(f'  {label:>5}: {ms_per_token:6.2f} ms/token')
    return ms_per_token

latencies = {}

print('Benchmarking fp16...')
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
latencies['fp16'] = bench(model_fp16, 'fp16')
del model_fp16; torch.cuda.empty_cache()

print('Benchmarking int8...')
model_int8 = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_int8, device_map='cuda')
latencies['int8'] = bench(model_int8, 'int8')
del model_int8; torch.cuda.empty_cache()

print('Benchmarking nf4...')
model_nf4 = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_nf4, device_map='cuda')
latencies['nf4'] = bench(model_nf4, 'nf4')

Benchmarking fp16...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


   fp16:  18.74 ms/token
Benchmarking int8...
   int8:  69.47 ms/token
Benchmarking nf4...
    nf4:  34.58 ms/token


In [6]:
lab.check(3)

OK — generation latency (ms/token, lower is better):
   fp16: 18.74 ms/token
   int8: 69.47 ms/token
    nf4: 34.58 ms/token
STEP_PASSED


Step 3 Complete! Scroll down to continue...

True

## Step 4 — Quality: does quantization actually preserve accuracy?

We measure **perplexity** (from Lab 8) on a held-out English snippet. For quantization to be useful, perplexity degradation has to be small. The LLM.int8() and QLoRA papers both claim near-zero-degradation under their respective schemes; let's verify.

Expected pattern:

- fp16: baseline
- int8: ~1–3% worse
- nf4: ~2–5% worse (tradeoff for the ~4× memory saving)

If you see much more degradation on nf4, `bnb_4bit_compute_dtype=torch.bfloat16` sometimes helps on Ampere+ GPUs.

On our 3060 Ti fp16 tends to be fine. bf16 is an alternative — some people swear by it for training stability.

### Set up the perplexity eval

Perplexity = exp(cross-entropy loss) — the standard language-model quality metric. Lower is better. We use a ~350-token English passage so the signal is large enough to distinguish quantized variants from fp16 (a 1% ppl bump on a 50-token sequence is noise; on 350 tokens it's meaningful). We'll score the *same* passage across fp16, int8, and nf4 to see the precision-quality tradeoff in one shot.

In [8]:
import math

# A ~350-token English passage for the eval
eval_text = (
    'Deep learning has transformed how we build machine learning systems. '
    'Rather than hand-engineering features, we now train neural networks on massive datasets '
    'and let them discover useful representations on their own. Transformers in particular, '
    'introduced in the Attention Is All You Need paper in 2017, have become the dominant architecture '
    'for language, vision, and speech tasks. Fine-tuning, quantization, and efficient serving '
    'are the three engineering levers that let small teams deploy these models at scale. '
    'Quantization in particular converts 32-bit floating-point weights into lower-precision representations '
    'like 8-bit integers or 4-bit normal-float values. The smaller footprint means more of the model '
    'fits in fast GPU memory, and since memory bandwidth is the bottleneck during autoregressive generation, '
    'lower-precision weights often mean faster inference as well.'
)

input_ids = tokenizer(eval_text, return_tensors='pt').input_ids.to('cuda')
print(f'Eval text: {input_ids.shape[1]} tokens')

@torch.no_grad()
def measure_ppl(m, ids):
    m.eval()
    out = m(ids, labels=ids)
    return math.exp(out.loss.item())

Eval text: 184 tokens


### Measure across all three precisions

We load one model at a time, measure perplexity, free VRAM, then load the next. Loading all three simultaneously would OOM a consumer GPU. fp16 is the baseline — int8 typically adds <1% ppl; nf4 adds 1–3%. The tradeoff is 2-4× less VRAM.

In [9]:
ppl_nf4 = measure_ppl(model_nf4, input_ids)
print(f'  nf4:  ppl = {ppl_nf4:.3f}')
del model_nf4; torch.cuda.empty_cache()

print('Reloading int8 for perplexity...')
model_int8 = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_int8, device_map='cuda')
ppl_int8 = measure_ppl(model_int8, input_ids)
print(f'  int8: ppl = {ppl_int8:.3f}')
del model_int8; torch.cuda.empty_cache()

print('Reloading fp16 for perplexity...')
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
ppl_fp16 = measure_ppl(model_fp16, input_ids)
print(f'  fp16: ppl = {ppl_fp16:.3f}  (baseline)')

print(f'\n=== Quality vs baseline ===')
print(f'  int8: {100*(ppl_int8/ppl_fp16-1):+.2f}%')
print(f'  nf4 : {100*(ppl_nf4/ppl_fp16-1):+.2f}%')

  nf4:  ppl = 11.214
Reloading int8 for perplexity...
  int8: ppl = 10.940
Reloading fp16 for perplexity...
  fp16: ppl = 11.026  (baseline)

=== Quality vs baseline ===
  int8: -0.78%
  nf4 : +1.70%


In [10]:
lab.check(4)

OK — perplexity on held-out text: fp16=11.03, int8=10.94 (-0.8%), nf4=11.21 (+1.7%)
STEP_PASSED


Step 4 Complete! Lab complete!

True

---

## What you just built

A production-shaped quantization evaluation: three loading strategies, VRAM/latency/quality table across them. This is the exact benchmark methodology you'd run before committing to a quantization scheme for a production deployment.

## What to read next

- **[LLM.int8() paper (Dettmers et al., 2022)](https://arxiv.org/abs/2208.07339)** — the mixed-precision outlier trick.
- **[QLoRA paper (Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314)** — NF4, double-quantization, and the fine-tuning recipe.
- **[GPTQ paper (Frantar et al., 2022)](https://arxiv.org/abs/2210.17323)** and **[AWQ paper (Lin et al., 2023)](https://arxiv.org/abs/2306.00978)** — calibration-based 4-bit methods with better accuracy than bitsandbytes-4bit, worth it for production deployments.
- **[bitsandbytes docs](https://huggingface.co/docs/bitsandbytes/main/en/index)** — all the config options, when to use each.
- **[Tim Dettmers — 8-bit quantization blog](https://timdettmers.com/2022/08/17/llm-int8-and-emergent-features/)** — the story behind the outlier discovery.
- **[TensorRT-LLM](https://github.com/NVIDIA/TensorRT-LLM)** — NVIDIA's production serving stack with the best INT8/FP8 kernels for H100/A100. What you'd use to deploy this at scale.

## What to try next

- Install `auto-gptq` and benchmark a pre-quantized **GPTQ** model (e.g. `TheBloke/TinyLlama-1.1B-Chat-v1.0-GPTQ`). GPTQ typically beats bitsandbytes-4bit on perplexity.
- Try `bnb_4bit_compute_dtype=torch.bfloat16` on an Ampere+ GPU; check whether quality or throughput improves.
- Scale up: load **Qwen2.5-7B** in NF4 on your 8GB GPU — it should just fit, unreachable in fp16.
- Combine: Load a 7B base in NF4, add LoRA adapters (Lab 1) trained in fp16 on top. That's the full QLoRA recipe.